<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 6 · How Should My Fund Trade?

Your Studio 5 fund is on the class leaderboard. Ms. Womack picked its mix once, and since then it has just sat there.
Real fund managers have to answer one more question, and tonight you answer it for **your** fund:

> **How should my fund trade, and do my rules help?**

| Scenario | What it does | The idea behind it |
|---|---|---|
| 🅰️ **Set and forget** | fit on the year before Jan 2, 2026, then hold all of 2026 | your Studio 5 fund's idea |
| 🅱️ **Trade back monthly** | the same starting mix, reset at the start of every month | discipline: trim the winners, top up the losers |
| 🅲 **Refit quarterly** | rerun Ms. Womack every quarter on the latest year of data | learn as you go |

Each scenario runs **with and without your rules**, after a small **trading cost**, on every trading day of 2026 so far.

### Tonight: four wins, in order

| | Win | What you do | About |
|---|---|---|---|
| 1 | **Load my fund** | paste one link from your Studio 5 receipt | 10 min |
| 2 | **My dashboard** | Run all; your three scenarios appear on one page | 15 min |
| 3 | **My twist** | change ONE thing and see what it does | 30 min |
| 4 | **My call** | read the market like an investor (bull, bear or sideways), pick a scenario and send it | 20 min |

**This notebook is also your final project.** Between now and Friday, Dec 4 you add a second twist, rerun it after the
Dec 3 close, and write a 2-page report. Every number you need is printed for you, and the last cell makes a one-page
fact sheet and a README for your resume. Everything is written already: you change the dictionaries.

## Import modules (download Pyomo, ipopt, Bonmin and yfinance)

Same setup as Studio 5. Run this first, every time. About a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt and Bonmin)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
import json
import time
import requests
from pyomo.environ import *
from IPython.display import display
import matplotlib.dates as mdates

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solvers
ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

# Win 1 · Load my fund

Your Studio 5 receipt printed a line called **your private record**: a long link that ends in `.json`. Paste it
between the quotes below. The notebook reads your stock list, your rules and your extra rules from it, exactly as you
sent them, so there is nothing to retype. (Keep the link to yourself: it is your fund's private record.)

**No link?** Leave `record_link` blank and choose your stocks the Studio 5 way with `my_theme`.

In [ ]:
record_link = ''      # paste the "your private record" link from your Studio 5 receipt

# only used when record_link is blank (the same choices as Studio 5):
#   'starter', a theme ('connecticut', 'ai_chips', 'oil_and_gas', 'banks_and_payments', 'dividends', 'healthcare',
#   'food_and_restaurants', 'travel', 'defense_and_aerospace', 'utilities_and_power', 'shopping',
#   'sports_and_entertainment'), 'draw' or 'dartboard'
my_theme = 'starter'
my_netid = ''         # your NetID ('draw' and 'dartboard' use it)

Run the next cell as it is. It holds the Studio 5 stock lists and the Studio 5 default rules, then loads your fund.

In [ ]:
# the Studio 5 starter list and the twelve themes (every company was in the S&P 500 on Dec 31, 2025)
starter = {
    'AAPL': 'Technology', 'MSFT': 'Technology', 'NVDA': 'Technology', 'GOOGL': 'Communication Services',
    'META': 'Communication Services', 'JPM': 'Financials', 'GS': 'Financials', 'V': 'Financials',
    'JNJ': 'Health Care', 'LLY': 'Health Care', 'UNH': 'Health Care', 'XOM': 'Energy', 'CVX': 'Energy',
    'KO': 'Consumer Staples', 'WMT': 'Consumer Staples', 'COST': 'Consumer Staples',
    'AMZN': 'Consumer Discretionary', 'HD': 'Consumer Discretionary', 'CAT': 'Industrials', 'LMT': 'Industrials',
}
themes = {
    'connecticut': ['TRV', 'HIG', 'CI', 'SWK', 'RTX', 'OTIS', 'CARR', 'URI', 'SYF', 'HUBB', 'ES', 'IT', 'FDS', 'CHTR', 'XOM', 'PG', 'MSFT', 'UNH'],
    'ai_chips': ['NVDA', 'AMD', 'AVGO', 'MU', 'LRCX', 'AMAT', 'KLAC', 'INTC', 'QCOM', 'TXN', 'ADI', 'ANET', 'MSFT', 'GOOGL', 'META', 'ORCL', 'LLY', 'XOM'],
    'oil_and_gas': ['XOM', 'CVX', 'COP', 'EOG', 'SLB', 'OXY', 'PSX', 'MPC', 'VLO', 'KMI', 'WMB', 'OKE', 'HAL', 'DVN', 'FANG', 'MSFT', 'JNJ'],
    'banks_and_payments': ['JPM', 'BAC', 'WFC', 'C', 'GS', 'MS', 'SCHW', 'V', 'MA', 'AXP', 'PYPL', 'COF', 'BLK', 'USB', 'AAPL', 'UNH', 'CVX'],
    'dividends': ['KO', 'PEP', 'PG', 'JNJ', 'MRK', 'ABBV', 'VZ', 'T', 'DUK', 'SO', 'XOM', 'CVX', 'MMM', 'IBM', 'MO', 'PM', 'CSCO', 'O'],
    'healthcare': ['LLY', 'JNJ', 'UNH', 'ABBV', 'MRK', 'PFE', 'TMO', 'ABT', 'ISRG', 'AMGN', 'GILD', 'CVS', 'MDT', 'BSX', 'SYK', 'AAPL', 'XOM'],
    'food_and_restaurants': ['MCD', 'SBUX', 'CMG', 'YUM', 'DPZ', 'KO', 'PEP', 'MDLZ', 'HSY', 'GIS', 'KHC', 'KDP', 'DRI', 'TSN', 'COST', 'AAPL', 'JNJ', 'XOM'],
    'travel': ['BKNG', 'MAR', 'HLT', 'ABNB', 'EXPE', 'DAL', 'UAL', 'LUV', 'RCL', 'CCL', 'NCLH', 'LVS', 'WYNN', 'MGM', 'UBER', 'AAPL', 'JNJ', 'XOM'],
    'defense_and_aerospace': ['LMT', 'RTX', 'NOC', 'GD', 'BA', 'LHX', 'HII', 'TXT', 'GE', 'HON', 'TDG', 'HWM', 'PLTR', 'LDOS', 'XOM', 'JNJ', 'MSFT'],
    'utilities_and_power': ['NEE', 'DUK', 'SO', 'D', 'AEP', 'EXC', 'SRE', 'XEL', 'PEG', 'ED', 'CEG', 'VST', 'ETR', 'FSLR', 'AES', 'XOM', 'JNJ', 'MSFT'],
    'shopping': ['AMZN', 'WMT', 'COST', 'TGT', 'HD', 'LOW', 'TJX', 'NKE', 'ROST', 'ULTA', 'DG', 'DLTR', 'BBY', 'EBAY', 'AAPL', 'CVS', 'XOM'],
    'sports_and_entertainment': ['DIS', 'NFLX', 'LYV', 'NKE', 'TTWO', 'CMCSA', 'CHTR', 'MAR', 'RCL', 'LVS', 'FOXA', 'HAS', 'GOOGL', 'META', 'AAPL', 'JNJ', 'XOM'],
}

# every S&P 500 company's sector (technology is called 'Technology', as in Studio 5)
members = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/sp500_2025-12-31.csv')
sp500_sectors = {}
for i in range(len(members)):
    sector = members['sector'].iloc[i]
    if sector == 'Information Technology':
        sector = 'Technology'
    sp500_sectors[members['ticker'].iloc[i]] = sector

# the Studio 5 default rules (a record replaces them with the rules you actually sent)
rules = {
    'min_share': 0.02,                                            # if you invest at all, at least 2%
    'max_share': 0.10,                                            # never more than 10% in one stock
    'must_have_sectors': ['Technology', 'Health Care', 'Energy'],  # at least one stock from each
    'max_stocks': None,                                           # e.g. 12, or None for no limit
}
extra_rules = {
    'sector_cap': None,     # e.g. 0.30: at most 30% of the money in any one sector
    'min_stocks': None,     # e.g. 12: hold at least 12 stocks
    'must_hold': [],        # e.g. ['AAPL']: these must be held (y = 1)
    'never_hold': [],       # e.g. ['XOM']: these can never be held (y = 0)
    'either_or': [],        # e.g. [['XOM', 'CVX']]: hold one or the other, not both
    'if_then': [],          # e.g. [['NVDA', 'AMD']]: if you hold NVDA, you must hold AMD
}

# a number from your NetID, so a draw or a dartboard is always the same for you
seed = 0
for letter in my_netid.strip().lower():
    seed += ord(letter)

record = None
loaded = None
if record_link.strip() != '':
    if record_link.strip().startswith('http'):
        record = requests.get(record_link.strip(), timeout=30).json()
    else:
        record = json.load(open(record_link.strip()))
    for entry in record['entries']:
        if isinstance(entry.get('sectors'), dict) and len(entry['sectors']) > 0:
            loaded = entry          # the latest entry with a stock list wins

if loaded is not None:
    my_stocks = loaded['sectors']
    if isinstance(loaded.get('rules'), dict):
        for key in loaded['rules']:
            rules[key] = loaded['rules'][key]
    if isinstance(loaded.get('extra_rules'), dict):
        for key in loaded['extra_rules']:
            extra_rules[key] = loaded['extra_rules'][key]
    my_theme = loaded.get('stock_theme') or 'custom'
    nickname_on_board = record.get('nickname', '')
    print("Loaded your fund from the record:", nickname_on_board)
elif record_link.strip() != '':
    print("That record has no stock list in it (it may be from before Nov 4). Using my_theme instead.")
    nickname_on_board = record.get('nickname', '')
else:
    nickname_on_board = ''

if loaded is None:
    if my_theme == 'starter':
        picked = list(starter)
    elif my_theme == 'draw':
        my_theme = sorted(themes)[seed % len(themes)]
        picked = themes[my_theme]
        print("Your NetID drew the theme:", my_theme)
    elif my_theme == 'dartboard':
        picked = list(members['ticker'].sample(20, random_state=seed))
        print("The monkey threw 20 darts at the S&P 500.")
    elif my_theme in themes:
        picked = themes[my_theme]
    else:
        print("Unknown theme:", my_theme, "- using the starter list")
        picked = list(starter)
    my_stocks = {}
    for t in picked:
        my_stocks[t] = starter.get(t, sp500_sectors.get(t, 'Other'))

print()
print("stock list:", my_theme, "|", len(my_stocks), "stocks:", ", ".join(my_stocks))
print("rules:", rules)
switched_on = {}
for key in extra_rules:
    if extra_rules[key] not in [None, []]:
        switched_on[key] = extra_rules[key]
print("extra rules switched on:", switched_on if len(switched_on) > 0 else "none")

### Daily prices

Daily adjusted closing prices from October 2024 to the latest close, for your stocks and the S&P 500 (SPY). Daily,
not monthly like Studio 5: rebalancing happens on a calendar, so we need every trading day. The model needs a year of
memory before Jan 2, 2026, which is why we start in October 2024. If Yahoo blocks Colab, the cell uses the class copy.

In [ ]:
import yfinance as yf

saved_prices = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/daily_prices.csv', index_col=0, parse_dates=True)


def get_prices(tickers):
    # daily closes for these tickers (+ SPY) from Oct 2024; the class copy fills in anything Yahoo will not give us
    downloaded = {}
    failed = []
    for ticker in list(tickers) + ['SPY']:
        try:
            data = yf.download(ticker, start='2024-10-01', interval='1d', auto_adjust=True, progress=False)
            if len(data) > 250:
                downloaded[ticker] = data['Close'].squeeze()
            else:
                failed.append(ticker)
        except Exception as problem:
            failed.append(ticker)
    table = pd.DataFrame(downloaded)
    if 'SPY' not in table.columns or len(table.columns) < 0.8 * (len(tickers) + 1):
        print("Yahoo did not give us enough stocks, so we are using the class copy instead.")
        table = pd.DataFrame(index=saved_prices.index)
        failed = []
        for ticker in list(tickers) + ['SPY']:
            if ticker in saved_prices.columns:
                table[ticker] = saved_prices[ticker]
            else:
                failed.append(ticker)
    if len(failed) > 0:
        print("No prices for:", failed)
    # today's row is not a close until the market has closed, so we stop at yesterday
    table = table[table.index < pd.Timestamp.today().normalize()]
    return table


prices = get_prices(my_stocks)
returns = prices.pct_change().iloc[1:]
days = returns.loc['2026-01-01':].index     # every trading day of 2026 so far
print(prices.shape[0], "trading days for", prices.shape[1] - 1, "stocks + SPY, through", prices.index[-1].date())
print("2026 so far:", len(days), "trading days, from", days[0].date(), "to", days[-1].date())

# The engine (run it, no edits)

Three cells that do all of the work. You do not need to change anything in them, but here is what they do.

**`solve_portfolio`** is Studio 5's model, with one change: instead of five years of monthly returns, it gets a
**window**, the last `memory_days` trading days *before* the day it trades. It never sees the day itself.

**The risk limit** each time: the risk of an even split of the same stocks over the same window. In words: *be as
risky as simply spreading the money evenly, but earn the most you can for that risk.* (Studio 5's Step 6 yardstick.)

**Quality control** checks every Bonmin answer against every rule as it runs, just like Studio 5.

**Trading days.** The market is open Monday to Friday, minus holidays: about 21 trading days a month and 252 a year.
"Monthly" means the first trading day of each month; "quarterly" means the first trading day of January, April, July
and October.

In [ ]:
tolerance = 1e-6
qc_log = []


def solve_portfolio(window, use_rules, fund, blocked):
    # Studio 5's model on a window of daily returns; blocked stocks get no money (the momentum filter)
    names = list(window.columns)
    sectors = fund['sectors']
    fund_rules = fund['rules']
    fund_extra = fund['extra_rules']
    df_return = window.mean()
    df_cov = window.cov()

    # the risk limit: the risk of an even split over the same days
    risk_limit = 0
    for i in names:
        for j in names:
            risk_limit += df_cov.loc[i, j] / len(names) ** 2

    model = ConcreteModel()
    model.x = Var(names, domain=NonNegativeReals, bounds=(0, 1))
    for s in names:
        model.x[s] = 1 / len(names)

    obj_expr = 0
    for s in names:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()
    budget = 0
    for s in names:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    risk_expr = 0
    for i in names:
        for j in names:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    for s in blocked:
        model.constraints.add(model.x[s] == 0)

    if use_rules:
        model.y = Var(names, domain=Binary)
        for s in names:
            model.y[s] = 1
        for s in names:
            if fund_rules['max_share'] is not None:
                model.constraints.add(model.x[s] <= fund_rules['max_share'] * model.y[s])
            else:
                model.constraints.add(model.x[s] <= model.y[s])
            if fund_rules['min_share'] is not None:
                model.constraints.add(model.x[s] >= fund_rules['min_share'] * model.y[s])
        for sector in fund_rules['must_have_sectors']:
            picks = 0
            count = 0
            for s in names:
                if sectors.get(s) == sector and s not in blocked:
                    picks += model.y[s]
                    count += 1
            if count > 0:    # a sector you have no (unblocked) stocks in is skipped
                model.constraints.add(picks >= 1)
        if fund_rules['max_stocks'] is not None:
            held = 0
            for s in names:
                held += model.y[s]
            model.constraints.add(held <= fund_rules['max_stocks'])
        if fund_extra['sector_cap'] is not None:
            for sector in set(sectors.values()):
                in_sector = 0
                count = 0
                for s in names:
                    if sectors.get(s) == sector:
                        in_sector += model.x[s]
                        count += 1
                if count > 0:
                    model.constraints.add(in_sector <= fund_extra['sector_cap'])
        if fund_extra['min_stocks'] is not None:
            held = 0
            for s in names:
                held += model.y[s]
            model.constraints.add(held >= fund_extra['min_stocks'])
        for s in fund_extra['must_hold']:
            if s in names:
                model.constraints.add(model.y[s] == 1)
        for s in fund_extra['never_hold']:
            if s in names:
                model.constraints.add(model.y[s] == 0)
        for pair in fund_extra['either_or']:
            if pair[0] in names and pair[1] in names:
                model.constraints.add(model.y[pair[0]] + model.y[pair[1]] <= 1)
        for pair in fund_extra['if_then']:
            if pair[0] in names and pair[1] in names:
                model.constraints.add(model.y[pair[0]] <= model.y[pair[1]])
        solver = SolverFactory('bonmin', executable=bonmin_path)
        solver.options['bonmin.time_limit'] = 60
        results = solver.solve(model)
    else:
        results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    if results.solver.termination_condition not in [TerminationCondition.optimal, TerminationCondition.maxTimeLimit]:
        return None
    weights = {}
    held = {}
    for s in names:
        weights[s] = max(0.0, model.x[s]())
        if use_rules:
            held[s] = model.y[s]()
    if use_rules:
        qc_log.append(check_rules(weights, held, fund, value(risk_expr), risk_limit))
    total = 0
    for s in names:
        if weights[s] < 0.0001:
            weights[s] = 0
        total += weights[s]
    for s in names:
        weights[s] = weights[s] / total
    return weights


def check_rules(weights, held, fund, risk, risk_limit):
    # every broken promise in one Bonmin answer (an empty list means it passed)
    problems = []
    held_count = 0
    fund_rules = fund['rules']
    for s in weights:
        x = weights[s]
        y = held[s]
        if abs(y - round(y)) > tolerance:
            problems.append("%s: y = %.6f is not a clean 0 or 1" % (s, y))
        if round(y) == 0 and x > tolerance:
            problems.append("%s: y = 0 but x = %.6f" % (s, x))
        if round(y) == 1:
            held_count += 1
            if fund_rules['min_share'] is not None and x < fund_rules['min_share'] - tolerance:
                problems.append("%s: x = %.4f is under the floor" % (s, x))
            if fund_rules['max_share'] is not None and x > fund_rules['max_share'] + tolerance:
                problems.append("%s: x = %.4f is over the cap" % (s, x))
    total = 0
    for s in weights:
        total += weights[s]
    if abs(total - 1) > 10 * tolerance:
        problems.append("the shares add up to %.6f, not 1" % total)
    if risk > risk_limit + 10 * tolerance:
        problems.append("the risk is over the limit")
    if fund_rules['max_stocks'] is not None and held_count > fund_rules['max_stocks']:
        problems.append("%d stocks held, more than %d" % (held_count, fund_rules['max_stocks']))
    return problems


print("solve_portfolio and check_rules are ready")

### The settings: one dictionary for all three scenarios

These are the class settings. Leave them as they are for now: changing one of them is a **twist** (Win 3).

In [ ]:
settings = {
    'memory_days': 252,             # the model looks back this many trading days (252 = about a year)
    'trade_back_every_months': 1,   # B: trade back to the starting mix every month
    'refit_every_months': 3,        # C: re-fit Ms. Womack every 3 months (a quarter)
    'trading_cost': 0.001,          # 0.1% of every dollar bought or sold (trade $4,000, pay $4)
    'momentum_filter': False,       # True: at every fit, stocks below their 200-day average get no money
}

scenario_names = {
    'A': 'A · set and forget',
    'B': 'B · trade back monthly',
    'C': 'C · refit quarterly',
}

In [ ]:
def first_days(every_months):
    # the first trading day of every month (1), every quarter (3), every half year (6), ... in 2026 so far
    picked = []
    last_month = None
    for day in days:
        if day.month != last_month:
            if (day.month - 1) % every_months == 0:
                picked.append(day)
            last_month = day.month
    return picked


def below_200_day(day, names):
    # the momentum filter: stocks whose last close before day is under their 200-day average
    history = prices.loc[prices.index < day, names].tail(200)
    blocked = []
    for s in names:
        if history[s].iloc[-1] < history[s].mean():
            blocked.append(s)
    return blocked


def fit(day, use_rules, fund, s):
    # Ms. Womack's mix for one day, using only the memory_days before it (remembered, so it is solved once)
    key = (day, use_rules, fund['name'], s['memory_days'], s['momentum_filter'])
    if key in fund['solved']:
        return fund['solved'][key]
    names = fund['stocks']
    window = returns.loc[returns.index < day, names].tail(s['memory_days'])
    blocked = []
    if s['momentum_filter']:
        blocked = below_200_day(day, names)
        for t in fund['extra_rules']['must_hold']:
            if t in blocked:
                blocked.remove(t)
    weights = solve_portfolio(window, use_rules, fund, blocked)
    if weights is None and len(blocked) > 0:
        print("  %s: too many stocks under their 200-day average for your rules, so the filter is off that day" % day.date())
        weights = solve_portfolio(window, use_rules, fund, [])
    if weights is None:
        print("  %s: no portfolio keeps every rule, so this one has no rules" % day.date())
        weights = solve_portfolio(window, False, fund, [])
    fund['solved'][key] = weights
    return weights


def run_scenario(kind, use_rules, fund, s):
    # $10,000 on the first trading day of 2026, then ride the prices, trading only on the scenario's days
    names = fund['stocks']
    start_mix = fit(days[0], use_rules, fund, s)
    if kind == 'A':
        trade_days = []
    elif kind == 'B':
        trade_days = first_days(s['trade_back_every_months'])[1:]
    else:
        trade_days = first_days(s['refit_every_months'])[1:]

    holdings = {}
    traded = 0.0
    for t in names:
        traded += 10000 * start_mix[t]
    worth = 10000 - s['trading_cost'] * traded
    for t in names:
        holdings[t] = worth * start_mix[t]
    path = {days[0]: worth}
    mixes = {days[0]: start_mix}
    for k in range(1, len(days)):
        day = days[k]
        worth = 0
        for t in names:
            holdings[t] = holdings[t] * (1 + returns.loc[day, t])
            worth += holdings[t]
        if day in trade_days:
            if kind == 'C':
                target = fit(day, use_rules, fund, s)
            else:
                target = start_mix
            trade = 0
            for t in names:
                trade += abs(worth * target[t] - holdings[t])
            traded += trade
            worth = worth - s['trading_cost'] * trade
            for t in names:
                holdings[t] = worth * target[t]
            mixes[day] = target
        path[day] = worth
    return {'path': pd.Series(path), 'traded': traded, 'costs': s['trading_cost'] * traded, 'mixes': mixes,
            'trades': len(trade_days)}


def make_fund(name, stock_sectors, fund_rules, fund_extra):
    # everything one version of your fund needs (the stocks that have prices, their sectors and the rules)
    names = []
    for t in stock_sectors:
        if t in returns.columns and returns[t].isna().sum() == 0:
            names.append(t)
        else:
            print("  leaving out", t, "(no full price history)")
    return {'name': name, 'stocks': names, 'sectors': stock_sectors, 'rules': fund_rules, 'extra_rules': fund_extra,
            'solved': {}}


def run_all(fund, s):
    started = time.time()
    results = {}
    for kind in ['A', 'B', 'C']:
        for use_rules in [True, False]:
            results[(kind, use_rules)] = run_scenario(kind, use_rules, fund, s)
    print("%s: 3 scenarios x with and without your rules, %d stocks, in %.0f seconds"
          % (fund['name'], len(fund['stocks']), time.time() - started))
    return results


spy_path = 10000 * prices.loc[days, 'SPY'] / prices.loc[days[0], 'SPY']
print("the engine is ready")

# Win 2 · My dashboard

Run the next three cells. The first runs your fund through all three scenarios (about a minute), the second checks
every Bonmin answer, and the third draws your dashboard.

In [ ]:
my_fund = make_fund('my fund', my_stocks, rules, extra_rules)
base = run_all(my_fund, settings)


def worst_drop(path):
    return 100 * float((1 - path / path.cummax()).max())


def scorecard(results):
    rows = []
    for kind in ['A', 'B', 'C']:
        for use_rules in [True, False]:
            run = results[(kind, use_rules)]
            rows.append({
                'scenario': scenario_names[kind],
                'rules': 'with your rules' if use_rules else 'no rules',
                '$10,000 became': round(float(run['path'].iloc[-1]), 0),
                'total return (%)': round(100 * float(run['path'].iloc[-1] / 10000 - 1), 1),
                'worst drop (%)': round(worst_drop(run['path']), 1),
                'trades': run['trades'],
                'trading costs ($)': round(run['costs'], 0),
                'vs S&P 500 ($)': round(float(run['path'].iloc[-1] - spy_path.iloc[-1]), 0),
            })
    return pd.DataFrame(rows)


base_card = scorecard(base)
print("S&P 500 (SPY) over the same days: $10,000 became $%.0f" % spy_path.iloc[-1])
base_card

In [ ]:
failed = 0
for problems in qc_log:
    if len(problems) > 0:
        failed += 1
        for problem in problems[:3]:
            print("problem:", problem)
if failed == 0:
    print("QUALITY CONTROL PASSED: %d Bonmin solves, every one keeps every rule." % len(qc_log))
else:
    print("QUALITY CONTROL: %d of %d Bonmin solves broke a rule (listed above). Tell Dr. Dave." % (failed, len(qc_log)))

In [ ]:
colors = {'A': 'tab:blue', 'B': 'tab:orange', 'C': 'tab:green'}


def dashboard(results, fund, title_text):
    fig, panels = subplots(2, 2, figsize=(14, 9))

    # 1. growth of $10,000, with your rules
    panel = panels[0, 0]
    for kind in ['A', 'B', 'C']:
        path = results[(kind, True)]['path']
        panel.plot(path.index, path.values, color=colors[kind], linewidth=2, label=scenario_names[kind])
    panel.plot(spy_path.index, spy_path.values, color='black', linewidth=2.5, label='S&P 500 (SPY)')
    panel.axhline(10000, color='lightgray', linestyle=':')
    panel.set_title('1 · $10,000 on Jan 2, 2026, with your rules, after costs')
    panel.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    panel.set_ylabel('value ($)')
    panel.legend(fontsize=9, loc='upper left')

    # 2. how far below its best day each one fell
    panel = panels[0, 1]
    for kind in ['A', 'B', 'C']:
        path = results[(kind, True)]['path']
        panel.plot(path.index, -100 * (1 - path / path.cummax()), color=colors[kind], label=scenario_names[kind])
    panel.plot(spy_path.index, -100 * (1 - spy_path / spy_path.cummax()), color='black', linewidth=2, label='S&P 500')
    panel.set_title('2 · Below its best day so far (%)')
    panel.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    panel.set_ylabel('% below the high')
    panel.legend(fontsize=9, loc='lower left')

    # 3. where each one ended: with your rules vs no rules
    panel = panels[1, 0]
    place = np.arange(3)
    ruled = []
    free = []
    costs = []
    for kind in ['A', 'B', 'C']:
        ruled.append(results[(kind, True)]['path'].iloc[-1])
        free.append(results[(kind, False)]['path'].iloc[-1])
        costs.append(results[(kind, True)]['costs'])
    panel.bar(place - 0.2, ruled, 0.4, color=[colors['A'], colors['B'], colors['C']], label='with your rules')
    panel.bar(place + 0.2, free, 0.4, color='lightgray', edgecolor='gray', label='no rules')
    panel.axhline(spy_path.iloc[-1], color='black', linewidth=2, label='S&P 500')
    for k in range(3):
        panel.text(place[k] - 0.2, ruled[k], '$%.0f\ncosts $%.0f' % (ruled[k], costs[k]), ha='center', va='bottom', fontsize=8)
    panel.set_xticks(place)
    panel.set_xticklabels([scenario_names['A'], scenario_names['B'], scenario_names['C']])
    low = min(min(ruled), min(free), spy_path.iloc[-1])
    high = max(max(ruled), max(free), spy_path.iloc[-1])
    panel.set_ylim(low - 0.15 * (high - low) - 200, high + 0.25 * (high - low) + 200)
    panel.set_title('3 · Where each one ended (latest close)')
    panel.legend(fontsize=9, loc='upper left')

    # 4. what C held after each refit (the 8 stocks it used most)
    panel = panels[1, 1]
    mixes = pd.DataFrame(results[('C', True)]['mixes']).T * 100
    top = mixes.max().sort_values(ascending=False).head(8).index
    bottom = np.zeros(len(mixes))
    labels = []
    for day in mixes.index:
        labels.append(day.strftime('%b %d'))
    for t in top:
        panel.bar(labels, mixes[t].values, bottom=bottom, label=t)
        bottom = bottom + mixes[t].values
    panel.bar(labels, 100 - bottom, bottom=bottom, color='lightgray', label='everything else')
    panel.set_title('4 · What C held after each refit (% of the money)')
    panel.set_ylabel('% of the money')
    panel.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5))

    fig.suptitle(title_text, fontsize=15, fontweight='bold')
    fig.tight_layout()
    return fig


if nickname_on_board != '':
    fund_title = nickname_on_board
else:
    fund_title = 'My fund (' + my_theme + ')'
fig = dashboard(base, my_fund, fund_title + ': how should it trade?')
show()

**How to read it.** Panel 1 is the race. Panel 2 is the pain: how far each one fell below its best day. Panel 3
is the finish line, with your rules (color) next to no rules (gray), and the trading costs each scenario paid. Panel 4
is what learning as you go looks like: how C's mix changed each quarter.

Three things to say out loud to your neighbor:
1. Which scenario is ahead, and by how much? Is it ahead of the S&P 500?
2. Did your rules help or hurt? In which scenario the most?
3. Did trading more (B and C) pay for its costs?

# Win 3 · My twist

Now make it yours. A **twist** is one change to your fund. The notebook reruns all three scenarios with the twist and
tells you what changed. **Change one thing at a time**, so you know what did it.

| Twist | Put this in `twist_1` | A question it answers |
|---|---|---|
| 🧺 **New stock list** | `'stock_list': 'oil_and_gas'` (any theme), or your own `['AAPL', 'MSFT', ...]` | Is it the stocks or the strategy? |
| 🔗 **Invent a rule** | `'extra_rules': {'either_or': [['XOM', 'CVX']]}` (or `if_then`, `sector_cap`, `must_hold`, `never_hold`) | What does a rule cost or earn? |
| 🎚️ **Loosen or tighten a rule** | `'rules': {'max_share': 0.25}` or `'rules': {'max_stocks': 8}` | Is the cap holding you back? |
| ⏱️ **Change the clock** | `'settings': {'refit_every_months': 1}` or `{'memory_days': 63}` | Does trading more help? |
| 💸 **Change the cost** | `'settings': {'trading_cost': 0.005}` | When does trading stop paying? |
| 📈 **Momentum filter** | `'settings': {'momentum_filter': True}` | Ms. Womack plus the 50/200 idea from the momentum bonus |
| 🃏 **Wild card** | anything else: ask Dr. Dave first | for when you have your own idea |

The example below is filled in (a looser cap: up to 20% in one stock instead of 10%), so you can see a twist work
before you write your own. Then
replace it with yours: give it a short name, change one thing, and run the two cells.

In [ ]:
twist_1 = {
    'name': 'looser cap',          # a short name for your twist
    'stock_list': None,            # a theme name, your own list of tickers, or None to keep yours
    'rules': {'max_share': 0.20},  # change a rule, e.g. {'max_share': 0.25}
    'extra_rules': {},             # add a linking rule, e.g. {'either_or': [['XOM', 'CVX']]}
    'settings': {},                # change the clock or the cost, e.g. {'refit_every_months': 1}
}

In [ ]:
def apply_twist(twist):
    # your fund with the twist: a new copy of the stocks, rules and settings, with the one change made
    global prices, returns
    stock_sectors = dict(my_stocks)
    if twist['stock_list'] is not None:
        if isinstance(twist['stock_list'], str):
            new_list = themes.get(twist['stock_list'], [])
            if len(new_list) == 0:
                print("Unknown theme:", twist['stock_list'])
        else:
            new_list = twist['stock_list']
        stock_sectors = {}
        for t in new_list:
            stock_sectors[t] = starter.get(t, sp500_sectors.get(t, 'Other'))
        missing = []
        for t in stock_sectors:
            if t not in prices.columns:
                missing.append(t)
        if len(missing) > 0:
            more = get_prices(missing)
            for t in more.columns:
                if t != 'SPY':
                    prices[t] = more[t]
            returns = prices.pct_change().iloc[1:]
    twist_rules = dict(rules)
    for key in twist['rules']:
        twist_rules[key] = twist['rules'][key]
    twist_extra = dict(extra_rules)
    for key in twist['extra_rules']:
        twist_extra[key] = twist['extra_rules'][key]
    twist_settings = dict(settings)
    for key in twist['settings']:
        twist_settings[key] = twist['settings'][key]
    return make_fund(twist['name'], stock_sectors, twist_rules, twist_extra), twist_settings


def what_changed(before, after, twist_name):
    rows = []
    for kind in ['A', 'B', 'C']:
        old = before[(kind, True)]['path']
        new = after[(kind, True)]['path']
        rows.append({
            'scenario (with your rules)': scenario_names[kind],
            'my fund ($)': round(float(old.iloc[-1]), 0),
            twist_name + ' ($)': round(float(new.iloc[-1]), 0),
            'change ($)': round(float(new.iloc[-1] - old.iloc[-1]), 0),
            'worst drop before (%)': round(worst_drop(old), 1),
            'worst drop after (%)': round(worst_drop(new), 1),
            'trading costs after ($)': round(after[(kind, True)]['costs'], 0),
        })
    table = pd.DataFrame(rows)
    for k in range(len(table)):
        change = table['change ($)'].iloc[k]
        if change > 0:
            print("%s: the twist helped, by $%.0f" % (table['scenario (with your rules)'].iloc[k], change))
        elif change < 0:
            print("%s: the twist hurt, by $%.0f" % (table['scenario (with your rules)'].iloc[k], -change))
        else:
            print("%s: the twist changed nothing (same portfolio)" % table['scenario (with your rules)'].iloc[k])
    return table


def compare_plot(before, after, twist_name):
    figure(figsize=(12, 5))
    for kind in ['A', 'B', 'C']:
        old = before[(kind, True)]['path']
        new = after[(kind, True)]['path']
        plot(old.index, old.values, color=colors[kind], linewidth=1.5, alpha=0.6, label=scenario_names[kind] + ', my fund')
        plot(new.index, new.values, color=colors[kind], linewidth=2.5, linestyle='--', label=scenario_names[kind] + ', ' + twist_name)
    plot(spy_path.index, spy_path.values, color='black', linewidth=2, label='S&P 500')
    axhline(10000, color='lightgray', linestyle=':')
    ylabel('value of $10,000')
    title('My fund (solid) vs my twist "' + twist_name + '" (dashed), with your rules, after costs')
    legend(fontsize=8, loc='upper left', ncol=2)
    show()


twist_fund_1, twist_settings_1 = apply_twist(twist_1)
twisted_1 = run_all(twist_fund_1, twist_settings_1)
compare_plot(base, twisted_1, twist_1['name'])
what_changed(base, twisted_1, twist_1['name'])

**Write it down in one sentence:** *"My twist (___) helped / hurt scenario ___ by \$___, and I think that is because
___."* That sentence goes straight into your final report.

**Twist 2 is for the final project.** Fill it in any time before Dec 4 (it is skipped while `name` is blank). Make it
a different kind of twist from your first one, so your report has two different ideas in it.

In [ ]:
twist_2 = {
    'name': '',             # blank = skipped (fill it in for the final project)
    'stock_list': None,
    'rules': {},
    'extra_rules': {},
    'settings': {},
}

twisted_2 = None
if twist_2['name'].strip() == '':
    print("Twist 2 is empty: fill it in for the final project (a different kind of twist from twist 1).")
else:
    twist_fund_2, twist_settings_2 = apply_twist(twist_2)
    twisted_2 = run_all(twist_fund_2, twist_settings_2)
    compare_plot(base, twisted_2, twist_2['name'])
    display(what_changed(base, twisted_2, twist_2['name']))

# What should an investor do now?

Time to switch hats. You are on the **investment committee**, and the question is about the next two weeks, not the
past ten months: **is the market in a bull, a bear, or going sideways?** Nobody knows for sure. Investors look at a
few simple signals, and so will you. Every signal uses only prices through the latest close: no peeking.

| Signal | 🐂 bull if... | 🐻 bear if... |
|---|---|---|
| **Trend** | the S&P 500 is above its 200-day average | it is below it |
| **Crossover** | the 50-day average is above the 200-day average (a "golden cross") | it is below (a "death cross") |
| **Momentum** | the S&P 500 is up more than 3% over the last 3 months (63 trading days) | down more than 3% |
| **Calm or stormy** | daily ups and downs are calmer than usual for the past year | stormier than usual (25% or more above normal) |
| **Distance from the high** | less than 5% below its high of the past year | more than 10% below it |
| **Breadth (your stocks)** | more than 60% of your stocks are above their own 200-day average | fewer than 40% are |

Anything in between counts as ➖ sideways. The 50 and 200-day averages are the same ones from the momentum bonus
notebook.

In [ ]:
signals_from = prices.index[-1] - pd.Timedelta(days=400)
spy = prices['SPY']
average_50 = spy.rolling(50).mean()
average_200 = spy.rolling(200).mean()
daily_spy = spy.pct_change()
stormy = daily_spy.rolling(21).std() * np.sqrt(252) * 100           # the last month's ups and downs, per year (%)
normal = stormy.tail(252).median()
high_252 = spy.rolling(252, min_periods=120).max()
below_high = 100 * (1 - spy / high_252)

breadth_rows = {}
for t in my_fund['stocks']:
    breadth_rows[t] = 100 * (prices[t].iloc[-1] / prices[t].tail(200).mean() - 1)
breadth = pd.Series(breadth_rows).sort_values()
share_above = 100 * (breadth > 0).mean()
momentum_3m = 100 * (spy.iloc[-1] / spy.iloc[-64] - 1)


def read(bull, bear):
    # one signal's reading
    if bull:
        return '🐂 bull'
    if bear:
        return '🐻 bear'
    return '➖ sideways'


signals = pd.DataFrame([
    {'signal': 'Trend', 'reading': 'S&P 500 %.0f vs 200-day average %.0f' % (spy.iloc[-1], average_200.iloc[-1]),
     'call': read(spy.iloc[-1] > average_200.iloc[-1], spy.iloc[-1] < average_200.iloc[-1])},
    {'signal': 'Crossover', 'reading': '50-day %.0f vs 200-day %.0f' % (average_50.iloc[-1], average_200.iloc[-1]),
     'call': read(average_50.iloc[-1] > average_200.iloc[-1], average_50.iloc[-1] < average_200.iloc[-1])},
    {'signal': 'Momentum', 'reading': '%+.1f%% over the last 63 trading days' % momentum_3m,
     'call': read(momentum_3m > 3, momentum_3m < -3)},
    {'signal': 'Calm or stormy', 'reading': 'ups and downs %.0f%% a year vs %.0f%% normal' % (stormy.iloc[-1], normal),
     'call': read(stormy.iloc[-1] < normal, stormy.iloc[-1] > 1.25 * normal)},
    {'signal': 'Distance from the high', 'reading': '%.1f%% below its 1-year high' % below_high.iloc[-1],
     'call': read(below_high.iloc[-1] < 5, below_high.iloc[-1] > 10)},
    {'signal': 'Breadth (your stocks)', 'reading': '%.0f%% of your stocks above their 200-day average' % share_above,
     'call': read(share_above > 60, share_above < 40)},
])

fig, panels = subplots(2, 2, figsize=(14, 8.5))
panel = panels[0, 0]
panel.plot(spy.loc[signals_from:].index, spy.loc[signals_from:].values, color='black', label='S&P 500 (SPY)')
panel.plot(average_50.loc[signals_from:].index, average_50.loc[signals_from:].values, color='tab:orange', label='50-day average')
panel.plot(average_200.loc[signals_from:].index, average_200.loc[signals_from:].values, color='tab:purple', label='200-day average')
panel.set_title('Trend and crossover')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
panel.legend(fontsize=9)
panel = panels[0, 1]
panel.plot(stormy.loc[signals_from:].index, stormy.loc[signals_from:].values, color='tab:red')
panel.axhline(normal, color='gray', linestyle='--', label='normal (median of the past year)')
panel.axhline(1.25 * normal, color='tab:red', linestyle=':', label='stormy (25% above normal)')
panel.set_title('Calm or stormy: the last month of ups and downs (% a year)')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
panel.legend(fontsize=9)
panel = panels[1, 0]
panel.fill_between(below_high.loc[signals_from:].index, 0, -below_high.loc[signals_from:].values, color='tab:red', alpha=0.4)
panel.axhline(-5, color='gray', linestyle='--', label='5% below')
panel.axhline(-10, color='tab:red', linestyle=':', label='10% below')
panel.set_title('Distance from the 1-year high (%)')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
panel.legend(fontsize=9, loc='lower left')
panel = panels[1, 1]
bar_colors = []
for gap in breadth.values:
    if gap > 0:
        bar_colors.append('tab:green')
    else:
        bar_colors.append('tab:red')
panel.barh(breadth.index, breadth.values, color=bar_colors)
panel.axvline(0, color='black')
panel.set_title('Breadth: each of your stocks vs its own 200-day average (%)')
fig.suptitle('What should an investor do now? (through the close of %s)' % prices.index[-1].date(), fontsize=14, fontweight='bold')
fig.tight_layout()
show()

bulls = int((signals['call'] == '🐂 bull').sum())
bears = int((signals['call'] == '🐻 bear').sum())
if bulls - bears >= 2:
    lean = '🐂 the data leans BULL'
elif bears - bulls >= 2:
    lean = '🐻 the data leans BEAR'
else:
    lean = '➖ the data is MIXED (sideways)'
print("%d bull, %d bear, %d sideways signals: %s" % (bulls, bears, 6 - bulls - bears, lean))
signals

### The playbook

Rules of thumb that investors use, not laws. Your call can disagree with the signals, as long as you can say why.

| Your outlook | What investors often do | The scenario that fits | A twist that fits |
|---|---|---|---|
| 🐂 **Bull** (the market keeps climbing) | stay fully invested and let the winners run | 🅰️ set and forget | loosen the cap (`'max_share': 0.20`) |
| ➖ **Sideways** (up and down, going nowhere) | buy the dips and sell the pops inside the range | 🅱️ trade back monthly | trade back more often |
| 🐻 **Bear** (the market falls) | play defense: drop what is falling, lean on calmer stocks | 🅲 refit quarterly | the momentum filter, or a sector cap |

One honest note: a real investor who turns bearish can sell everything and hold cash. Our funds are always fully
invested, so here being defensive means choosing *which* stocks to hold, not whether to hold any.

# Win 4 · My call

Make your call for the next two weeks, from tonight's close (Nov 18) to the close on **Thursday, Dec 3**:

* **`outlook`**: bull, bear or sideways. It is scored against the S&P 500 over those two weeks: up more than 1% is a
  bull, down more than 1% is a bear, anything in between is sideways.
* **`scenario`**: the one your fund runs from now on (A, B or C).
* **`use_twist`**: `True` runs the scenario with your twist 1.
* **`why`**: one sentence. It goes on your fund page.

In [ ]:
my_call = {
    'outlook': 'sideways',  # 'bull', 'bear' or 'sideways': the S&P 500 from Nov 18 to Dec 3
    'scenario': 'B',        # 'A', 'B' or 'C': how your fund trades from now on
    'use_twist': False,     # True: run the scenario with your twist 1
    'why': '',              # one sentence: why this call?
}

In [ ]:
if my_call['use_twist']:
    call_fund = twist_fund_1
    call_settings = twist_settings_1
    call_results = twisted_1
else:
    call_fund = my_fund
    call_settings = settings
    call_results = base
kind = my_call['scenario']
call_name = scenario_names[kind]
if my_call['use_twist']:
    call_name = call_name + ' + ' + twist_1['name']


def holdings_today(use_rules):
    # what the scenario holds after the latest close
    names = call_fund['stocks']
    start_mix = fit(days[0], use_rules, call_fund, call_settings)
    if kind == 'A':
        grown = {}
        total = 0
        for t in names:
            grown[t] = start_mix[t] * prices[t].iloc[-1] / prices.loc[days[0], t]
            total += grown[t]
        for t in names:
            grown[t] = grown[t] / total
        return grown
    if kind == 'B':
        return start_mix
    return fit(prices.index[-1] + pd.Timedelta(days=1), use_rules, call_fund, call_settings)


now_ruled = holdings_today(True)
now_free = holdings_today(False)
now = pd.DataFrame({'no rules (%)': pd.Series(now_free) * 100, 'with your rules (%)': pd.Series(now_ruled) * 100})
now = now[(now['no rules (%)'] > 0.05) | (now['with your rules (%)'] > 0.05)]
now['sector'] = pd.Series(call_fund['sectors'])
print("your call:", my_call['outlook'], "|", call_name)
print("2026 so far, after costs: with your rules $%.0f, no rules $%.0f, S&P 500 $%.0f" % (
    call_results[(kind, True)]['path'].iloc[-1], call_results[(kind, False)]['path'].iloc[-1], spy_path.iloc[-1]))
now.sort_values('with your rules (%)', ascending=False).round(1)

### Send it to the leaderboard

Same as Studio 5: fill in the lines, run the next cell, and sign in when Colab asks. This counts as a **change** to
your Studio 5 fund: it starts from what your fund is worth at the next close, and your call and your reason are saved
in its history. **Use the same NetID and nickname as in Studio 5.**

In [ ]:
netid = ""            # your UConn NetID, for example abc12345
nickname = ""         # the SAME nickname as in Studio 5
theme = ""            # one line: what your fund is about now
class_code = ""       # Dr. Dave writes it on the board

if nickname.strip() == '' and nickname_on_board != '':
    nickname = nickname_on_board
reason = "Studio 6: %s | outlook %s | %s" % (call_name, my_call['outlook'], my_call['why'].strip())
print("nickname:", nickname)
print("reason:", reason)

In [ ]:
SUBMIT_URL = 'https://us-central1-opim5641-portfolio-f26.cloudfunctions.net/submit-portfolio'
BOARD_URL = 'https://storage.googleapis.com/opim5641-leaderboard-f26/index.html'


def shares_to_send(weights):
    kept = {}
    for t in weights:
        if weights[t] > 0.001:
            kept[t] = round(float(weights[t]), 6)
    return kept


# the 2026 backtest of the scenario you picked, one point a week (small enough to send)
even_path = 10000 * (prices.loc[days, call_fund['stocks']] / prices.loc[days[0], call_fund['stocks']]).mean(axis=1)
lines = {'with your rules': call_results[(kind, True)]['path'], 'no rules': call_results[(kind, False)]['path'],
         'equal split of your stocks': even_path, 'S&P 500 (SPY)': spy_path}
backtest_paths = {}
backtest_table = {}
for name in lines:
    path = lines[name]
    one_line = {}
    for k in range(0, len(path), 5):
        one_line[str(path.index[k].date())] = round(float(path.iloc[k]), 2)
    one_line[str(path.index[-1].date())] = round(float(path.iloc[-1]), 2)
    backtest_paths[name] = one_line
    backtest_table[name] = {'ending value ($)': round(float(path.iloc[-1]), 0),
                            'total return (%)': round(100 * float(path.iloc[-1] / 10000 - 1), 1),
                            'worst drop (%)': round(worst_drop(path), 1)}

entry = {
    'netid': netid.strip().lower(),
    'nickname': nickname.strip(),
    'theme': theme.strip(),
    'reason': reason.strip()[:200],
    'class_code': class_code.strip(),
    'portfolios': {'no_rules': shares_to_send(now_free), 'with_rules': shares_to_send(now_ruled)},
    'rules': call_fund['rules'],
    'extra_rules': call_fund['extra_rules'],
    'stock_theme': my_theme,
    'sectors': call_fund['sectors'],
    'outlook': my_call['outlook'],
    'call': {'scenario': kind, 'use_twist': my_call['use_twist'], 'why': my_call['why'].strip()[:200],
             'twist': twist_1 if my_call['use_twist'] else None},
    'strategy': {'name': call_name, 'settings': call_settings},
    'data_window': [str(returns.index[0].date()), str(returns.index[-1].date())],
    'backtest': backtest_table,
    'backtest_paths': backtest_paths,
    'backtest_title': "2026 so far: %s, $10,000 from Jan 2, after a %.1f%% trading cost" % (call_name, 100 * call_settings['trading_cost']),
    'backtest_note': "From the Studio 6 notebook: %s, run on every trading day of 2026 so far, starting with $10,000 "
                     "on Jan 2. The model only ever used data from before each trading day." % call_name,
    'notebook': 'studio6',
}

missing = []
for field in ['netid', 'nickname', 'theme', 'class_code']:
    if entry[field] == '':
        missing.append(field)
if my_call['outlook'] not in ['bull', 'bear', 'sideways']:
    missing.append("my_call['outlook'] (bull, bear or sideways)")

if len(missing) > 0:
    print("Not sent yet. Fill in:", ", ".join(missing), "then run this cell again.")
elif 'google.colab' not in sys.modules:
    print("Not sent: run this notebook in Google Colab to sign in and submit.")
else:
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    answer = requests.post(SUBMIT_URL, data=json.dumps(entry, default=str), timeout=60,
                           headers={'Authorization': 'Bearer ' + credentials.token, 'Content-Type': 'application/json'})
    receipt = answer.json()
    if receipt.get('ok'):
        received = pd.Timestamp(receipt['received_utc']).tz_convert('America/New_York')
        print("Your call is in! This is what the leaderboard SAVED:")
        print("  nickname:     ", receipt['nickname'])
        print("  NetID:        ", receipt['netid'], " (signed in as", receipt['google_email'] + ")")
        print("  reason:       ", receipt.get('reason', ''))
        print("  received:     ", received.strftime('%a %b %d, %I:%M %p'), "Eastern")
        print("  stocks held:  ", receipt['holdings']['with_rules'], "with your rules,", receipt['holdings']['no_rules'], "with no rules")
        print("  it counts from the next market close (4 PM Eastern on a weekday)")
        if receipt.get('fund_page'):
            print("  your fund's page:", receipt['fund_page'])
        if receipt.get('record'):
            print("  your private record:", receipt['record'], "(the same link as before: keep it)")
        print("  the leaderboard: ", BOARD_URL)
    else:
        print("Not accepted:", receipt.get('problem'))

### Studio 6 night: your turn

Answer in a text cell before you leave:

1. Which scenario won 2026 so far for your fund, with your rules? Did it beat the S&P 500?
2. Did your rules help or hurt? Where the most?
3. Your twist: what did you change, and what did it do (one sentence, with a dollar amount)?
4. Your call: bull, bear or sideways, which scenario, and why?

# The final project (individual, due Friday, Dec 4, 11:59 PM)

The final project is this notebook, finished. Four steps:

1. **Twist 2.** Fill in `twist_2` (a different kind of twist from twist 1).
2. **Rerun after the Dec 3 close.** Paste your record link, **Run all**, and keep your twists and your call the same.
   (Do not resend unless you really mean to change your fund.)
3. **Your report numbers.** The next cell prints every number your report needs, each with a sentence starter.
4. **The 2-page report** (a PDF on HuskyCT), with these five parts:

| Part | Length | What goes in it |
|---|---|---|
| 1 · My fund and my rules | ½ page | your stocks, your rules, and why you chose them |
| 2 · A vs B vs C | ½ page | your dashboard and the three numbers that matter |
| 3 · My two twists | ½ page | what each one changed, with the dollar amounts and one sentence each |
| 4 · The live race and my call | ¼ page | your fund since Studio 5 vs the S&P 500, and whether your call was right |
| 5 · What I would do on Jan 1, 2027 | ¼ page | one recommendation, in plain words |

Your grade does **not** depend on making money. It depends on how clearly you test and explain your choices.

**Awards (just for fun, announced after the Dec 3 close):** 🏁 **Race winner** (the most money; over five weeks that
is mostly luck), 🛡️ **Steadiest** (beat the S&P 500 with the smallest worst drop), 🔮 **Best call** (called the
market right, so anyone can win), and 🔬 **Best investigation** (Dr. Dave's pick from the reports).

In [ ]:
def live_race():
    # your fund on the leaderboard since Studio 5, from the private record (None without a record link)
    if record_link.strip() == '':
        return None
    if record_link.strip().startswith('http'):
        fresh = requests.get(record_link.strip(), timeout=30).json()
    else:
        fresh = json.load(open(record_link.strip()))
    if len(fresh['values']['dates']) < 2:
        return None
    race = pd.DataFrame({'my fund': fresh['values']['with_rules'], 'no-rules twin': fresh['values']['no_rules']},
                        index=pd.to_datetime(fresh['values']['dates']))
    spy_race = pd.Series(fresh['spy']['values'], index=pd.to_datetime(fresh['spy']['dates']))
    race['S&P 500'] = spy_race.reindex(race.index)
    return race


def rules_in_words(fund_rules, fund_extra):
    words = []
    if fund_rules['max_share'] is not None:
        words.append('at most %.0f%% in one stock' % (100 * fund_rules['max_share']))
    if fund_rules['min_share'] is not None:
        words.append('at least %.0f%% in any stock it holds' % (100 * fund_rules['min_share']))
    if len(fund_rules['must_have_sectors']) > 0:
        words.append('at least one stock in ' + ', '.join(fund_rules['must_have_sectors']))
    if fund_rules['max_stocks'] is not None:
        words.append('no more than %d stocks' % fund_rules['max_stocks'])
    for key in fund_extra:
        if fund_extra[key] not in [None, []]:
            words.append('%s: %s' % (key.replace('_', ' '), fund_extra[key]))
    return '; '.join(words)


race = live_race()
best_kind = base_card[base_card['rules'] == 'with your rules'].sort_values('$10,000 became', ascending=False)['scenario'].iloc[0]
rules_rows = base_card.pivot_table(index='scenario', columns='rules', values='$10,000 became')

print("REPORT NUMBERS (through the close of %s)" % prices.index[-1].date())
print()
print("Part 1 · My fund and my rules")
print("  \"My fund holds %d stocks from the %s list, with these rules: %s.\"" % (len(my_fund['stocks']), my_theme,
                                                                             rules_in_words(rules, extra_rules)))
print()
print("Part 2 · A vs B vs C (with your rules, after costs)")
for k in range(len(base_card)):
    row = base_card.iloc[k]
    if row['rules'] == 'with your rules':
        print("  \"%s turned $10,000 into $%.0f (worst drop %.1f%%, costs $%.0f).\"" % (
            row['scenario'], row['$10,000 became'], row['worst drop (%)'], row['trading costs ($)']))
print("  \"The S&P 500 turned $10,000 into $%.0f. The best scenario for my fund was %s.\"" % (spy_path.iloc[-1], best_kind))
for scenario in rules_rows.index:
    gap = rules_rows.loc[scenario, 'with your rules'] - rules_rows.loc[scenario, 'no rules']
    if gap >= 0:
        print("  \"In %s my rules helped, by $%.0f.\"" % (scenario, gap))
    else:
        print("  \"In %s my rules cost me $%.0f.\"" % (scenario, -gap))
print()
print("Part 3 · My two twists (scenario %s)" % my_call['scenario'])
twist_runs = [(twist_1, twisted_1), (twist_2, twisted_2)]
for twist, twisted in twist_runs:
    if twisted is None:
        print("  (twist 2 is still empty)")
        continue
    change = twisted[(my_call['scenario'], True)]['path'].iloc[-1] - base[(my_call['scenario'], True)]['path'].iloc[-1]
    if change > 0:
        print("  \"My twist '%s' helped %s, by $%.0f.\"" % (twist['name'], scenario_names[my_call['scenario']], change))
    elif change < 0:
        print("  \"My twist '%s' hurt %s, by $%.0f.\"" % (twist['name'], scenario_names[my_call['scenario']], -change))
    else:
        print("  \"My twist '%s' did not change %s.\"" % (twist['name'], scenario_names[my_call['scenario']]))
print()
print("Part 4 · The live race and my call")
if race is None:
    print("  (paste your record link at the top to get the live race)")
else:
    print("  \"Since %s my fund went from $10,000 to $%.0f; its no-rules twin to $%.0f; the S&P 500 to $%.0f.\"" % (
        race.index[0].date(), race['my fund'].iloc[-1], race['no-rules twin'].iloc[-1], race['S&P 500'].iloc[-1]))
    print("  \"My worst drop in the live race was %.1f%%.\"" % worst_drop(race['my fund']))
print("  \"On Nov 18 I called the market %s and ran %s, because %s\"" % (my_call['outlook'], call_name, my_call['why'] or '___'))
print()
print("CHECKLIST")
print("  [%s] twist 1 has a name" % ('x' if twist_1['name'].strip() != '' else ' '))
print("  [%s] twist 2 has a name" % ('x' if twist_2['name'].strip() != '' else ' '))
print("  [%s] record link pasted (live race numbers)" % ('x' if race is not None else ' '))
print("  [%s] my call has a reason" % ('x' if my_call['why'].strip() != '' else ' '))

# 🎓 Your resume kit

Something to be proud of, made from your own numbers. The next cell makes:

* **`my_fund_fact_sheet.png`**: one page about your fund (your stocks, your rules, A vs B vs C, the live race and
  your call). It is also the dashboard page for your report, and something you can show in an interview.
* **`README.md`**: a ready-made front page if you save this notebook to your own GitHub (File, Save a copy in GitHub).
* **A resume bullet** you can paste and adjust.

In Colab, the two files download automatically.

In [ ]:
fact = figure(figsize=(11, 8.5))
fact.suptitle(fund_title + ' · fund fact sheet', fontsize=18, fontweight='bold', x=0.05, ha='left')
fact.text(0.05, 0.905, '%d stocks (%s) | max %.0f%% in one stock | must hold: %s | trading cost %.1f%%' % (
    len(my_fund['stocks']), my_theme, 100 * (rules['max_share'] or 1), ', '.join(rules['must_have_sectors']) or 'none',
    100 * settings['trading_cost']), fontsize=10, color='dimgray')

growth = fact.add_axes([0.06, 0.42, 0.55, 0.43])
for kind in ['A', 'B', 'C']:
    path = base[(kind, True)]['path']
    growth.plot(path.index, path.values, color=colors[kind], linewidth=2, label=scenario_names[kind])
growth.plot(spy_path.index, spy_path.values, color='black', linewidth=2.5, label='S&P 500')
growth.axhline(10000, color='lightgray', linestyle=':')
growth.set_title('2026 backtest: $10,000, with my rules, after costs', fontsize=11)
growth.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
growth.legend(fontsize=8, loc='upper left')

table_box = fact.add_axes([0.66, 0.42, 0.31, 0.43])
table_box.axis('off')
cells = []
for kind in ['A', 'B', 'C']:
    run = base[(kind, True)]
    cells.append([scenario_names[kind].split(' · ')[1], '$%.0f' % run['path'].iloc[-1], '%.1f%%' % worst_drop(run['path'])])
cells.append(['S&P 500', '$%.0f' % spy_path.iloc[-1], '%.1f%%' % worst_drop(spy_path)])
board = table_box.table(cellText=cells, colLabels=['scenario', 'ended', 'worst drop'], loc='upper center',
                        colWidths=[0.5, 0.25, 0.25])
board.auto_set_font_size(False)
board.set_fontsize(10)
board.scale(1, 2.2)
call_text = 'My call (Nov 18): %s, %s' % (my_call['outlook'], call_name)
if my_call['why'].strip() != '':
    call_text = call_text + '\n"' + my_call['why'].strip()[:90] + '"'
table_box.text(0, 0.18, call_text, fontsize=9, wrap=True)

live = fact.add_axes([0.06, 0.07, 0.55, 0.27])
if race is not None:
    live.plot(race.index, race['my fund'], color='tab:green', linewidth=2, label='my fund (live)')
    live.plot(race.index, race['S&P 500'], color='black', linewidth=2, label='S&P 500')
    live.set_title('The live race on the class leaderboard', fontsize=11)
    live.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    live.legend(fontsize=8, loc='upper left')
else:
    live.axis('off')
    live.text(0, 0.5, 'Paste your record link at the top to add the live race.', fontsize=10)
notes = fact.add_axes([0.66, 0.07, 0.31, 0.27])
notes.axis('off')
notes.text(0, 0.95, 'Built with', fontsize=10, fontweight='bold', va='top')
notes.text(0, 0.8, 'Python, pandas, Pyomo, Bonmin, ipopt\nmixed-integer optimization with linking rules\n'
           'walk-forward backtest with trading costs\nOPIM 5641 · UConn · Fall 2026', fontsize=9, va='top')
fact.savefig('my_fund_fact_sheet.png', dpi=150)
show()

best_row = base_card[base_card['rules'] == 'with your rules'].sort_values('$10,000 became', ascending=False).iloc[0]
bullet = ("Built a mixed-integer portfolio optimizer in Python (Pyomo, Bonmin) with business-rule linking constraints; "
          "backtested three rebalancing strategies on 2026 daily data with trading costs (best: %s, %+.1f%%); "
          "managed a live fund against the S&P 500 on a class leaderboard." % (best_row['scenario'], best_row['total return (%)']))
readme = ("# " + fund_title + "\n\n"
          "A portfolio I built and managed in OPIM 5641 (Business Decision Modeling) at UConn, Fall 2026.\n\n"
          "## What it does\n"
          "* Picks a portfolio of %d stocks with Ms. Womack's model: the highest average return for a risk limit, "
          "solved with Pyomo (ipopt, and Bonmin for the yes/no rules).\n"
          "* Business rules as linking constraints: a %.0f%% cap per stock, a %.0f%% floor for any stock held, "
          "must-have sectors.\n"
          "* Tests three ways to trade through 2026 (set and forget, trade back monthly, refit quarterly) with a "
          "%.1f%% trading cost, using only data from before each trading day.\n\n"
          "## Results (2026 so far, $10,000 start)\n" % (len(my_fund['stocks']), 100 * (rules['max_share'] or 1),
                                                       100 * (rules['min_share'] or 0), 100 * settings['trading_cost']))
for kind in ['A', 'B', 'C']:
    readme += "* %s: $%.0f\n" % (scenario_names[kind], base[(kind, True)]['path'].iloc[-1])
readme += "* S&P 500: $%.0f\n\n" % spy_path.iloc[-1]
readme += "![fact sheet](my_fund_fact_sheet.png)\n"
open('README.md', 'w').write(readme)

print("RESUME BULLET (adjust it to sound like you):")
print("  " + bullet)
print()
print("saved: my_fund_fact_sheet.png and README.md")
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('my_fund_fact_sheet.png')
    files.download('README.md')

### 🤖 Prompts to try with Claude

Copy one into Claude (or another AI assistant) with your dashboard or your report numbers pasted in:

* *"Here is my portfolio dashboard. Explain to a non-finance friend why scenario B beat scenario A, in three sentences."*
* *"My twist made my fund \$300 worse in scenario C. Give me three possible reasons I could check in my notebook."*
* *"I called the market bearish. Play devil's advocate: what would a bull say about the same six signals?"*
* *"Here is my resume bullet. Make it shorter and stronger without claiming anything I did not do."*